# Step 9 — SageMaker Model Monitor

Detect data drift with hourly monitoring.

In [ ]:
import boto3
import sagemaker
from sagemaker.model_monitor import DefaultModelMonitor, CronExpressionGenerator
from sagemaker.model_monitor.dataset_format import DatasetFormat
from sagemaker.predictor import Predictor
from sagemaker.serializers import JSONSerializer
from sagemaker.deserializers import JSONDeserializer

BUCKET = "YOUR-S3-BUCKET-NAME"
REGION = "us-east-2"
ENDPOINT_NAME = "symptom-classifier-endpoint"

boto_session = boto3.Session(region_name=REGION)
session = sagemaker.Session(boto_session=boto_session, default_bucket=BUCKET)
role = sagemaker.get_execution_role()

predictor = Predictor(
    endpoint_name=ENDPOINT_NAME,
    sagemaker_session=session,
    serializer=JSONSerializer(),
    deserializer=JSONDeserializer(),
)

## Send test requests to generate captured data

In [ ]:
test_symptoms = [
    "I have severe headaches and sensitivity to light",
    "I have a persistent cough fever and difficulty breathing",
    "My joints are swollen and painful especially in the morning",
    "I feel very thirsty urinate frequently and feel tired",
    "I have chest pain and shortness of breath",
]
for symptom in test_symptoms:
    predictor.predict({"inputs": symptom})
    print(f"Sent: {symptom[:50]}...")
print("Test requests sent ✅")

## Create baseline and monitoring schedule

In [ ]:
monitor = DefaultModelMonitor(
    role=role,
    instance_count=1,
    instance_type="ml.m5.xlarge",
    volume_size_in_gb=20,
    max_runtime_in_seconds=3600,
    sagemaker_session=session,
)

# Create baseline from training data
monitor.suggest_baseline(
    baseline_dataset=f"s3://{BUCKET}/symptom-data/processed/train_processed.csv",
    dataset_format=DatasetFormat.csv(header=True),
    output_s3_uri=f"s3://{BUCKET}/model-monitor/baseline",
    wait=True,
    logs=False,
)
print("Baseline created ✅")

# Create hourly monitoring schedule
monitor.create_monitoring_schedule(
    monitor_schedule_name="symptom-monitor-schedule",
    endpoint_input=ENDPOINT_NAME,
    output_s3_uri=f"s3://{BUCKET}/model-monitor/reports",
    statistics=monitor.baseline_statistics(),
    constraints=monitor.suggested_constraints(),
    schedule_cron_expression=CronExpressionGenerator.hourly(),
    enable_cloudwatch_metrics=True,
)
print("Monitoring schedule created ✅ — runs every hour")